> **Version corrigée** — ce notebook contient le code complet de tous les exercices, exécuté de bout en bout, ainsi qu'un élément de réponse pour chaque question d'observation. La version étudiant (à compléter soi-même) est téléchargeable depuis la page du cours.

# Boosting, gradient boosting et stacking

*Notebook 9/9 — Introduction à l'apprentissage supervisé (L3 MIASHS → Master, Guillaume Metzler, Université Lyon 2).*

Le notebook précédent combinait des modèles *forts* appris indépendamment (bagging, forêts aléatoires) pour réduire la variance de l'ensemble. On s'intéresse ici à une logique très différente : le **boosting**, qui combine de façon **séquentielle** des **apprenants faibles**, chacun corrigeant les erreurs des précédents, pour réduire le biais.

On étudie successivement :

- **Adaboost**, l'algorithme de boosting historique, basé sur une repondération des exemples mal classés ;
- le **gradient boosting**, qui généralise l'idée à n'importe quelle fonction de perte dérivable en apprenant sur les *pseudo-résidus* ;
- le **stacking**, une troisième famille de méthodes ensemblistes qui apprend un **méta-modèle** combinant des modèles de base potentiellement très hétérogènes.


In [ ]:
# Imports communs a l'ensemble du notebook
import time

import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import (
    make_classification, make_moons, make_circles,
    load_wine, load_breast_cancer, load_digits,
)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier,
    StackingClassifier,
)
from sklearn.metrics import accuracy_score

RNG = 42
np.random.seed(RNG)
plt.rcParams["figure.dpi"] = 100


## 1. Boosting et Adaboost

Le bagging combine en parallèle des modèles forts, appris indépendamment. Le **boosting** procède tout autrement : on construit une *séquence* de classifieurs $h_1, h_2, \ldots$ à partir d'**apprenants faibles** — des modèles à peine meilleurs qu'un tirage aléatoire (souches de décision, arbres de profondeur 1 ou 2, SVM linéaire sur un problème non séparable...) — où chaque nouveau $h_{t+1}$ est appris de façon à corriger les erreurs commises par $h_1, \ldots, h_t$.

**Adaboost** (*Adaptive Boosting*, Freund & Schapire, 1999) est l'algorithme de boosting le plus connu. Chaque exemple $i$ possède un poids $w_i^{(t)}$, initialisé à $1/m$. À chaque round $t$ :

1. on apprend un classifieur faible $h_t$ sur $S$ pondéré par $w^{(t)}$ ;
2. on calcule son erreur pondérée $\displaystyle \varepsilon_t = \sum_{i=1}^m w_i^{(t)}\, \mathbb{1}_{\{h_t(x_i) y_i < 0\}}$ ;
3. on lui attribue le poids (la confiance) $\displaystyle \alpha_t = \frac12 \ln\left(\frac{1-\varepsilon_t}{\varepsilon_t}\right)$ ;
4. on met à jour les poids des exemples : $\displaystyle w_i^{(t+1)} = \frac{w_i^{(t)} \exp(-\alpha_t y_i h_t(x_i))}{Z_t}$, avec $Z_t = 2\sqrt{\varepsilon_t(1-\varepsilon_t)}$.

Cette mise à jour augmente le poids des exemples mal classés par $h_t$ et diminue celui des exemples bien classés. Après $T$ rounds, la prédiction finale est $H_T(x) = \mathrm{sign}\left(\sum_{t=1}^T \alpha_t h_t(x)\right)$.


In [ ]:
# --- Le coefficient alpha_t en fonction de l'erreur ponderee ------------
eps_range = np.linspace(0.01, 0.99, 400)
alpha_range = 0.5 * np.log((1 - eps_range) / eps_range)

plt.figure(figsize=(7, 5))
plt.plot(eps_range, alpha_range)
plt.axhline(0, color="gray", linewidth=0.8)
plt.axvline(0.5, color="gray", linestyle="--", linewidth=0.8, label=r"$\varepsilon_t = 0.5$")
plt.xlabel(r"Erreur pondérée $\varepsilon_t$")
plt.ylabel(r"Poids du classifieur $\alpha_t$")
plt.title(r"$\alpha_t = \frac{1}{2}\ln\left(\frac{1-\varepsilon_t}{\varepsilon_t}\right)$ en fonction de $\varepsilon_t$")
plt.legend()
plt.tight_layout()
plt.show()


$$ $$

**Question 1 :** Que devient $\alpha_t$ lorsque $\varepsilon_t$ est proche de $0$ ? Proche de $1$ ? Et exactement à $\varepsilon_t = 0.5$ ?

$$ $$

$$ $$

**Question 2 :** Un apprenant faible dont l'erreur pondérée dépasse 0.5 reçoit un alpha négatif. Que signifie concrètement ce coefficient négatif pour la contribution de ce classifieur à $H_T$ ?

$$ $$

*Éléments de réponse.* Quand $\varepsilon_t \to 0$ (classifieur presque parfait), $\alpha_t \to +\infty$ : on lui fait une confiance quasi absolue. Quand $\varepsilon_t \to 1$ (classifieur presque toujours faux), $\alpha_t \to -\infty$. À $\varepsilon_t = 0.5$ (pas mieux que le hasard), $\alpha_t = 0$ : le classifieur ne pèse pas dans $H_T$.

*Éléments de réponse.* Un alpha négatif inverse la contribution du classifieur dans la somme pondérée : Adaboost prend en compte l'opposé de sa décision. Un classifieur systématiquement mauvais (dans le sens inverse de la vérité) devient ainsi utile une fois sa prédiction retournée.

### Exercice 1 — Calcul de $\alpha_t$ pour plusieurs erreurs pondérées

Reprenez la formule $\alpha_t = \frac12\ln\left(\frac{1-\varepsilon_t}{\varepsilon_t}\right)$, cette fois sans tracer de courbe : calculez-la directement avec `numpy` pour une liste de valeurs de $\varepsilon_t$.


In [ ]:
epsilons = [0.02, 0.1, 0.25, 0.4, 0.5, 0.65, 0.85]
alphas = [0.5 * np.log((1 - eps) / eps) for eps in epsilons]

for eps, alpha in zip(epsilons, alphas):
    if eps < 0.5:
        verdict = "digne de confiance (poids positif)"
    elif eps == 0.5:
        verdict = "inutile (poids nul)"
    else:
        verdict = "à inverser (poids négatif)"
    print(f"epsilon = {eps:.2f}  ->  alpha = {alpha:+.4f}   [{verdict}]")

assert abs(alphas[4]) < 1e-9  # epsilon = 0.5 -> alpha = 0
assert alphas[0] > 0 and alphas[-1] < 0


### Évolution des poids au fil des rounds

Illustrons maintenant, sur un petit jeu de données jouet, comment les poids des exemples évoluent au fil des itérations d'Adaboost : on implémente ici "à la main" les quelques lignes de l'algorithme, identiques à celles utilisées en interne par `AdaBoostClassifier`, la taille d'un point étant proportionnelle à son poids $w_i^{(t)}$ **avant** l'apprentissage de $h_t$ à ce round.


In [ ]:
# --- Evolution des poids des exemples au cours des rounds d'Adaboost ------
X_toy, y_toy = make_classification(n_samples=90, n_features=2, n_redundant=0,
                                    n_clusters_per_class=1, class_sep=0.8,
                                    flip_y=0.07, random_state=13)
y_pm = np.where(y_toy == 0, -1, 1)  # Adaboost utilise des labels +-1

m = len(X_toy)
w = np.ones(m) / m

n_rounds = 5
fig, axes = plt.subplots(1, n_rounds, figsize=(4.2 * n_rounds, 4.2), sharex=True, sharey=True)

for t in range(n_rounds):
    ax = axes[t]
    sizes = 15 + 500 * (w / w.max())
    ax.scatter(X_toy[:, 0], X_toy[:, 1], c=y_pm, cmap="coolwarm", s=sizes, edgecolor="k", linewidth=0.5)

    stump = DecisionTreeClassifier(max_depth=1, random_state=0)
    stump.fit(X_toy, y_pm, sample_weight=w)
    pred = stump.predict(X_toy)
    incorrect = pred != y_pm
    eps_t = np.clip(np.sum(w[incorrect]), 1e-8, 1 - 1e-8)
    alpha_t = 0.5 * np.log((1 - eps_t) / eps_t)

    ax.set_title(f"Round {t + 1}\n" + rf"$\varepsilon_t={eps_t:.2f}$, $\alpha_t={alpha_t:.2f}$")
    ax.set_xlabel("$x_1$")
    if t == 0:
        ax.set_ylabel("$x_2$")

    w = w * np.exp(-alpha_t * y_pm * pred)
    w = w / w.sum()

plt.suptitle("Adaboost : la taille d'un point est proportionnelle à son poids $w_i^{(t)}$")
plt.tight_layout()
plt.show()


$$ $$

**Question 3 :** Quels points grossissent visiblement d'un round à l'autre ? Que représentent-ils ?

$$ $$

$$ $$

**Question 4 :** Le poids d'un même exemple peut-il diminuer à un round puis ré-augmenter au round suivant ? D'après la règle de mise à jour, pourquoi cela peut-il se produire ?

$$ $$

*Éléments de réponse.* Les points qui grossissent sont ceux mal classés par la souche apprise au round courant : ce sont les exemples les plus difficiles, sur lesquels le round suivant doit porter une attention particulière.

*Éléments de réponse.* Oui. Le poids d'un point ne dépend que de sa classification par la souche du round courant, or cette souche change à chaque round (elle est apprise sur une distribution de poids différente). Un point correctement classé à un round peut très bien retomber du mauvais côté d'un nouveau split au round suivant, et voir son poids remonter : l'évolution n'est pas monotone pour un exemple pris isolément.

### Utilisation de `AdaBoostClassifier`

En pratique, on utilise directement `AdaBoostClassifier` de `scikit-learn`, en lui passant l'apprenant faible via le paramètre `estimator`. Regardons comment l'exactitude évolue avec le nombre d'itérations `n_estimators`.


In [ ]:
# --- AdaBoostClassifier : exactitude en fonction du nombre d'iterations ---
X, y = make_moons(n_samples=350, noise=0.28, random_state=5)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=5)

n_estimators_list = [1, 5, 10, 20, 50, 100, 200, 400]
train_acc, test_acc = [], []

for n in n_estimators_list:
    ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                              n_estimators=n, random_state=5)
    ada.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, ada.predict(X_train)))
    test_acc.append(accuracy_score(y_test, ada.predict(X_test)))

plt.figure(figsize=(7, 5))
plt.plot(n_estimators_list, train_acc, marker="o", label="exactitude (train)")
plt.plot(n_estimators_list, test_acc, marker="o", label="exactitude (test)")
plt.xscale("log")
plt.xlabel("nombre d'itérations (n_estimators, échelle log)")
plt.ylabel("exactitude")
plt.title("AdaBoostClassifier sur make_moons : exactitude vs nombre d'itérations")
plt.legend()
plt.tight_layout()
plt.show()


$$ $$

**Question 5 :** À partir de combien d'itérations la performance sur le test se stabilise-t-elle environ ? Continuer d'ajouter des classifieurs faibles au-delà de ce point apporte-t-il encore un gain ?

$$ $$

*Éléments de réponse.* La performance progresse nettement dans les premières dizaines d'itérations puis se stabilise (le gain marginal devient très faible, voire nul) : au-delà d'un certain nombre de rounds, ajouter des classifieurs faibles supplémentaires n'améliore plus vraiment l'exactitude sur le test — Adaboost reste néanmoins assez peu sujet au sur-apprentissage même avec un grand nombre d'itérations.

### Exercice 2 — `AdaBoostClassifier` contre une souche isolée

Sur le principe de la démo précédente : montrez qu'un ensemble d'apprenants faibles vaut mieux qu'un seul.


In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

stump_seul = DecisionTreeClassifier(max_depth=1, random_state=42)
stump_seul.fit(X_train, y_train)
acc_stump = accuracy_score(y_test, stump_seul.predict(X_test))

ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1),
                          n_estimators=100, random_state=42)
ada.fit(X_train, y_train)
acc_ada = accuracy_score(y_test, ada.predict(X_test))

print(f"Exactitude d'une souche isolée      : {acc_stump:.3f}")
print(f"Exactitude d'Adaboost (100 souches) : {acc_ada:.3f}")

assert acc_ada >= acc_stump


## 2. Gradient boosting

Adaboost repose implicitement sur la perte exponentielle, ce qui ne convient pas à toutes les situations. Le **gradient boosting** (Friedman, 2000) généralise le principe du boosting à n'importe quelle fonction de perte $\ell$ dérivable, en menant une optimisation dans l'**espace des fonctions** plutôt que dans l'espace des paramètres.

À chaque itération $t$, le modèle combiné est mis à jour par $H_t = H_{t-1} + \alpha_t h_t$, où le nouvel apprenant faible $h_t$ est appris pour approcher les **pseudo-résidus**, c'est-à-dire l'opposé du gradient de la perte par rapport à la prédiction courante :

$$\tilde y_i = -\left.\frac{\partial \ell(y_i, H_{t-1}(x_i))}{\partial H_{t-1}(x_i)}\right., \qquad h_t = \arg\min_h \sum_{i=1}^m (\tilde y_i - h(x_i))^2.$$

Pour la **perte quadratique** $\ell(y, H(x)) = (y-H(x))^2$ (régression), le pseudo-résidu vaut simplement $\tilde y = 2(y - H_{t-1}(x))$ : à un facteur 2 près, l'écart classique entre la vraie valeur et la prédiction courante. Une fois $h_t$ appris, son poids $\alpha_t$ est choisi pour minimiser $\ell$ le long de cette direction, et le **taux d'apprentissage** (*shrinkage*, paramètre `learning_rate` de `scikit-learn`) vient généralement le pondérer davantage : plus il est faible, plus la convergence est lente mais moins on risque de sur-apprendre.


In [ ]:
# --- Pseudo-residus pour la perte quadratique, premier round ("a la main") ---
y_toy = np.array([4.0, 7.0, 9.0, 12.0, 15.0, 6.0, 10.0])
H0 = np.full_like(y_toy, y_toy.mean())  # H0 optimal pour la perte quadratique : la moyenne
pseudo_res = 2 * (y_toy - H0)

fig, ax = plt.subplots(figsize=(8, 4.5))
idx = np.arange(len(y_toy))
ax.bar(idx - 0.15, y_toy - H0, width=0.3, label=r"écart $y_i - H_0(x_i)$")
ax.bar(idx + 0.15, pseudo_res, width=0.3, label=r"pseudo-résidu $\tilde y_i = 2(y_i - H_0(x_i))$")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(idx)
ax.set_xlabel("Exemple $i$")
ax.set_title(r"Pseudo-résidus au premier round (perte quadratique), $H_0 = \bar y$")
ax.legend()
plt.tight_layout()
plt.show()

print(f"H0 (hypothèse initiale, constante) = {H0[0]:.3f}")
print("Pseudo-résidus :", np.round(pseudo_res, 3))


### Exercice 3 — Pseudo-résidus pour un modèle courant non constant

On considère maintenant un modèle courant $H_{t-1}$ qui n'est plus constant : chaque exemple a sa propre prédiction courante.


In [ ]:
y = np.array([3.0, 5.0, 2.0, 8.0, 10.0, 6.0])
Ht_moins_1 = np.array([2.5, 4.0, 3.0, 7.0, 8.5, 6.5])

pseudo_res = 2 * (y - Ht_moins_1)
print("Pseudo-résidus :", np.round(pseudo_res, 3))

pire = np.argmax(np.abs(pseudo_res))
print(f"Exemple sur lequel le modèle se trompe le plus : indice {pire} "
      f"(pseudo-résidu = {pseudo_res[pire]:+.2f})")

assert np.allclose(pseudo_res, [1.0, 2.0, -2.0, 2.0, 3.0, -1.0])
assert pire == 4


### `staged_predict` : suivre l'évolution de l'erreur

`GradientBoostingClassifier` propose `staged_predict`, qui renvoie, pour $t = 1, \ldots, T$, la prédiction du modèle $H_t$ obtenu après les $t$ premières itérations : cela permet de suivre l'évolution de l'erreur sans ré-entraîner le modèle à chaque $t$. Faisons-le varier conjointement avec le **taux d'apprentissage**.


In [ ]:
# --- staged_predict : courbes erreur train/test selon le taux d'apprentissage ---
X, y = make_classification(n_samples=600, n_features=20, n_informative=8, n_redundant=4,
                            flip_y=0.03, class_sep=0.9, random_state=11)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=11)

learning_rates = [0.01, 0.1, 1.0]
n_est_max = 300

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, lr in zip(axes, learning_rates):
    gb = GradientBoostingClassifier(n_estimators=n_est_max, max_depth=2,
                                     learning_rate=lr, random_state=11)
    gb.fit(X_train, y_train)
    train_err = [1 - accuracy_score(y_train, p) for p in gb.staged_predict(X_train)]
    test_err = [1 - accuracy_score(y_test, p) for p in gb.staged_predict(X_test)]

    ax.plot(range(1, n_est_max + 1), train_err, label="train")
    ax.plot(range(1, n_est_max + 1), test_err, label="test")
    ax.set_title(f"learning_rate = {lr}")
    ax.set_xlabel("nombre d'itérations $T$")

axes[0].set_ylabel("taux d'erreur (1 - exactitude)")
axes[0].legend()
plt.suptitle("Gradient boosting : effet du taux d'apprentissage sur les courbes d'erreur train/test")
plt.tight_layout()
plt.show()


$$ $$

**Question 6 :** Avec `learning_rate=1.0`, à quelle vitesse l'erreur d'entraînement s'annule-t-elle, et que devient l'écart avec l'erreur de test ?

$$ $$

$$ $$

**Question 7 :** Avec `learning_rate=0.01`, la convergence est plus lente. Quel est l'intérêt pratique de ralentir ainsi l'apprentissage, malgré le coût d'un plus grand nombre d'itérations nécessaires ?

$$ $$

*Éléments de réponse.* Avec un taux d'apprentissage élevé, l'erreur d'entraînement chute très vite (le modèle ajuste rapidement les pseudo-résidus, jusqu'à un score parfait) alors que l'erreur de test se stabilise beaucoup plus tôt puis peut légèrement remonter : l'écart train/test se creuse rapidement, signe de sur-apprentissage.

*Éléments de réponse.* Un taux d'apprentissage faible agit comme une forme de régularisation : chaque nouvel arbre ne corrige qu'une petite partie de l'erreur, ce qui limite le risque de sur-apprendre le bruit des données. On accepte une convergence plus lente (donc plus d'itérations, plus de temps de calcul) contre une meilleure généralisation.

### Exercice 4 — Nombre d'itérations optimal via `staged_predict`

Même mécanique que la démo précédente, sur un autre jeu de données, avec un seul taux d'apprentissage cette fois : à vous d'identifier le nombre d'itérations optimal.


In [ ]:
X, y = make_circles(n_samples=400, noise=0.2, factor=0.4, random_state=9)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=9)

gb = GradientBoostingClassifier(n_estimators=250, max_depth=2, learning_rate=0.1, random_state=9)
gb.fit(X_train, y_train)

train_err = np.array([1 - accuracy_score(y_train, p) for p in gb.staged_predict(X_train)])
test_err = np.array([1 - accuracy_score(y_test, p) for p in gb.staged_predict(X_test)])

plt.figure(figsize=(7, 5))
plt.plot(range(1, len(train_err) + 1), train_err, label="train")
plt.plot(range(1, len(test_err) + 1), test_err, label="test")
plt.xlabel("nombre d'itérations $T$")
plt.ylabel("taux d'erreur (1 - exactitude)")
plt.title("Gradient boosting sur make_circles : erreurs train/test")
plt.legend()
plt.tight_layout()
plt.show()

n_opt = np.argmin(test_err) + 1  # +1 car staged_predict commence a T=1
print(f"Nombre d'itérations minimisant l'erreur de test : {n_opt} "
      f"(erreur de test = {test_err[n_opt - 1]:.3f})")


### Exercice 5 (niveau Master) — `GridSearchCV` sur un `GradientBoostingClassifier`

Réglons maintenant simultanément plusieurs hyperparamètres par validation croisée.


In [ ]:
data = load_wine()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

param_grid = {
    "n_estimators": [50, 100, 200],
    "learning_rate": [0.01, 0.1, 0.3],
    "max_depth": [1, 2, 3],
}

grid = GridSearchCV(GradientBoostingClassifier(random_state=42), param_grid, cv=5)
grid.fit(X_train, y_train)

best_params = grid.best_params_
test_acc = accuracy_score(y_test, grid.best_estimator_.predict(X_test))

print("Meilleurs hyperparamètres :", best_params)
print(f"Exactitude sur le jeu de test : {test_acc:.3f}")


### Random Forest, Adaboost et Gradient Boosting : un comparatif

Comparons enfin les trois familles vues jusqu'ici — une méthode de bagging (forêt aléatoire) et deux méthodes de boosting — sur un même jeu de données, en mesurant aussi leur temps d'entraînement.


In [ ]:
# --- Random Forest vs Adaboost vs Gradient Boosting : performance et temps ---
data = load_breast_cancer()
Xb, yb = data.data, data.target
Xb_train, Xb_test, yb_train, yb_test = train_test_split(Xb, yb, test_size=0.3, random_state=42, stratify=yb)

models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "Adaboost": AdaBoostClassifier(n_estimators=200, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=200, random_state=42),
}

noms, accs, temps = [], [], []
for name, model in models.items():
    t0 = time.time()
    model.fit(Xb_train, yb_train)
    dt = time.time() - t0
    acc = accuracy_score(yb_test, model.predict(Xb_test))
    noms.append(name); accs.append(acc); temps.append(dt)
    print(f"{name:20s} : exactitude = {acc:.3f}  |  temps d'entraînement = {dt:.3f} s")

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].bar(noms, accs, color=["tab:blue", "tab:orange", "tab:green"])
axes[0].set_ylabel("Exactitude (test)")
axes[0].set_ylim(min(accs) - 0.03, 1.0)
axes[0].set_title("Exactitude")
axes[1].bar(noms, temps, color=["tab:blue", "tab:orange", "tab:green"])
axes[1].set_ylabel("Temps d'entraînement (s)")
axes[1].set_title("Temps d'entraînement")
plt.tight_layout()
plt.show()


$$ $$

**Question 8 :** Les trois méthodes obtiennent-elles des exactitudes très différentes sur ce jeu de données ? Laquelle est la plus rapide à entraîner, et pourquoi (pensez à la façon dont chaque famille construit ses modèles de base) ?

$$ $$

*Éléments de réponse.* Les trois méthodes obtiennent en général une exactitude très proche et élevée sur ce jeu de données relativement simple. La forêt aléatoire est en général la plus rapide à entraîner : ses arbres sont indépendants et pourraient être appris en parallèle, alors que le boosting (Adaboost, gradient boosting) apprend ses modèles de base de façon strictement séquentielle, chacun dépendant du résultat du précédent — ce qui interdit toute parallélisation entre les rounds.

## 3. Stacking

Le **stacking** (*stacked generalization*, Wolpert, 1992) combine des modèles très différemment du bagging et du boosting : les modèles de base peuvent être de nature complètement différente (un k-NN, un SVM, un arbre de décision, ...), ils sont appris **indépendamment** sur les mêmes données, et leur combinaison n'est pas une moyenne ou un vote pondéré itérativement, mais est **apprise** par un **méta-modèle**.

La procédure comporte deux étapes : (i) on apprend un ensemble de modèles de base $h_1, \ldots, h_T$ sur le jeu d'entraînement ; (ii) les prédictions de ces modèles, $(h_1(x), \ldots, h_T(x))$, servent de **nouvelles features** pour entraîner un méta-modèle. Pour éviter que le méta-modèle ne sur-apprenne les prédictions "trop optimistes" des modèles de base sur leurs propres données d'entraînement, on utilise en pratique une validation croisée à $k$ plis pour générer des prédictions hors échantillon — c'est ce que fait `StackingClassifier` via son paramètre `cv`.


In [ ]:
# --- StackingClassifier vs modeles de base pris isolement -----------------
data = load_wine()
Xw, yw = data.data, data.target
Xw_train, Xw_test, yw_train, yw_test = train_test_split(Xw, yw, test_size=0.3, random_state=42, stratify=yw)

knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))
svm = make_pipeline(StandardScaler(), SVC(probability=True, random_state=42))
tree = DecisionTreeClassifier(max_depth=4, random_state=42)

base_models = {"k-NN": knn, "SVM": svm, "Arbre de décision": tree}
base_acc = {}
for name, model in base_models.items():
    model.fit(Xw_train, yw_train)
    base_acc[name] = accuracy_score(yw_test, model.predict(Xw_test))

stack = StackingClassifier(
    estimators=[("knn", knn), ("svm", svm), ("tree", tree)],
    final_estimator=LogisticRegression(max_iter=2000),
    cv=5,
)
stack.fit(Xw_train, yw_train)
stack_acc = accuracy_score(yw_test, stack.predict(Xw_test))

for name, acc in base_acc.items():
    print(f"{name:20s} : exactitude = {acc:.3f}")
print(f"{'Stacking':20s} : exactitude = {stack_acc:.3f}")

plt.figure(figsize=(7, 4.5))
noms = list(base_acc.keys()) + ["Stacking"]
valeurs = list(base_acc.values()) + [stack_acc]
plt.bar(noms, valeurs, color=["tab:blue", "tab:blue", "tab:blue", "tab:red"])
plt.ylabel("Exactitude (test)")
plt.ylim(min(valeurs) - 0.05, 1.02)
plt.title("StackingClassifier vs modèles de base pris isolément (load_wine)")
plt.tight_layout()
plt.show()


$$ $$

**Question 9 :** Le StackingClassifier fait-il mieux que le meilleur des trois modèles de base pris isolément ? D'où vient, selon vous, ce gain éventuel ?

$$ $$

*Éléments de réponse.* Le stacking obtient en général une exactitude au moins aussi bonne que le meilleur modèle de base, et souvent légèrement meilleure : le méta-modèle apprend à exploiter les complémentarités entre modèles de nature différente (k-NN, SVM, arbre), qui commettent des erreurs sur des exemples différents — là où le meilleur modèle isolé, aussi bon soit-il, se trompe toujours sur les mêmes exemples.

### Exercice 6 — Stacking sur un autre jeu de données

Même mécanique que la démo précédente, sur un jeu de données réel différent.


In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))
svm = make_pipeline(StandardScaler(), SVC(probability=True, random_state=42))
tree = DecisionTreeClassifier(max_depth=4, random_state=42)

base_models = {"k-NN": knn, "SVM": svm, "Arbre de décision": tree}
base_acc = {}
for name, model in base_models.items():
    model.fit(X_train, y_train)
    base_acc[name] = accuracy_score(y_test, model.predict(X_test))

stack = StackingClassifier(
    estimators=[("knn", knn), ("svm", svm), ("tree", tree)],
    final_estimator=LogisticRegression(max_iter=2000),
    cv=5,
)
stack.fit(X_train, y_train)
stack_acc = accuracy_score(y_test, stack.predict(X_test))

for name, acc in base_acc.items():
    print(f"{name:20s} : exactitude = {acc:.3f}")
print(f"{'Stacking':20s} : exactitude = {stack_acc:.3f}")


### Exercice 7 (niveau Master) — Stacking d'ensembles hétérogènes

Rien n'empêche d'utiliser, comme modèles de base d'un stacking, des méthodes ensemblistes elles-mêmes (Adaboost, gradient boosting) plutôt que des modèles simples : le stacking sait combiner des modèles de nature arbitraire.


In [ ]:
digits = load_digits()
mask = digits.target < 5
X, y = digits.data[mask], digits.target[mask]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

ada = AdaBoostClassifier(n_estimators=50, random_state=42)
gb = GradientBoostingClassifier(n_estimators=50, max_depth=2, random_state=42)
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))

base_models = {"Adaboost": ada, "Gradient Boosting": gb, "k-NN": knn}
base_acc = {}
for name, model in base_models.items():
    model.fit(X_train, y_train)
    base_acc[name] = accuracy_score(y_test, model.predict(X_test))

stack = StackingClassifier(
    estimators=[("ada", ada), ("gb", gb), ("knn", knn)],
    final_estimator=LogisticRegression(max_iter=2000),
    cv=5,
)
stack.fit(X_train, y_train)
stack_acc = accuracy_score(y_test, stack.predict(X_test))

for name, acc in base_acc.items():
    print(f"{name:20s} : exactitude = {acc:.3f}")
print(f"{'Stacking':20s} : exactitude = {stack_acc:.3f}")

print("\nMême les modèles de base les plus performants (des ensembles de boosting)")
print("peuvent encore être combinés par un méta-modèle : le stacking n'est pas réservé")
print("aux modèles simples.")


## Conclusion

Adaboost et le gradient boosting partagent le même principe séquentiel — corriger, à chaque round, les erreurs des rounds précédents — mais reposent sur des mécanismes différents : repondération explicite des exemples pour Adaboost, apprentissage des pseudo-résidus d'une perte quelconque pour le gradient boosting. Le stacking, lui, ne construit pas de séquence : il apprend simplement à combiner, via un méta-modèle, des modèles de base appris indépendamment, aussi hétérogènes soient-ils.

Avec le bagging et les forêts aléatoires vus dans le notebook précédent, ces méthodes ensemblistes — en particulier les forêts aléatoires et le gradient boosting — comptent aujourd'hui parmi les algorithmes les plus utilisés en apprentissage supervisé sur données tabulaires.
